In [6]:
import torch
print(torch.__version__)

import onnxscript
print(onnxscript.__version__)

import onnxruntime
print(onnxruntime.__version__)

2.8.0+cu128
0.5.6
1.23.2


In [8]:

import onnxruntime as ort

session = ort.InferenceSession('hair_classifier_v1.onnx', providers=["CPUExecutionProvider"])

inputs = session.get_inputs()
outputs = session.get_outputs()

input_name = inputs[0].name
output_name = outputs[0].name

print(f"Input name: {input_name}")
print(f"Output name: {output_name}")

Input name: input
Output name: output


In [9]:
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

In [14]:
url = 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'
img = download_image(url)
img = prepare_image(img, target_size=(200, 200))

In [18]:
# convert image to numpy array
import numpy as np
img_array = np.array(img).astype('float32')
img_array[0][0]

array([ 61., 104.,  22.], dtype=float32)

In [20]:
from torchvision import transforms
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

img_tensor = train_transforms(img)


In [22]:
# first pixel
img_tensor[:, 0, 0]

tensor([-1.0733, -0.2150, -1.4210])

In [24]:
result = session.run([output_name], {input_name: img_tensor.unsqueeze(0).numpy()})

In [25]:
result

[array([[0.09156641]], dtype=float32)]

REPOSITORY                                    TAG                    IMAGE ID       CREATED         SIZE
agrigorev/model-2025-hairstyle                v1                     4528ad1525d5   5 days ago      608MB


In [4]:
!curl -X POST \
  http://localhost:8123/2015-03-31/functions/function/invocations \
  -H "Content-Type: application/json" \
  -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'

[-0.0719808042049408]